# TASTF: Tier-Aware Spatiotemporal Forecasting for HetNets

This notebook implements the **TASTF** (Tier-Aware Spatiotemporal Forecasting) model for Heterogeneous Cellular Networks (HetNets). 

### Core Contribution
TASTF uses a **Heterogeneous Graph Convolutional Network** that treats Macro, Pico, and Femto base stations as distinct node types with type-specific convolutional layers via PyTorch Geometric `HeteroConv`. This approach explicitly models the hierarchy and cross-tier influence in HetNets, which is then processed by a **Transformer Encoder** to capture long-range temporal dependencies.

---

## 1. Environment Setup & Dependencies

In [ ]:
!pip install torch-geometric torch-scatter torch-sparse pandas numpy scikit-learn matplotlib

  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached torch_scatter-2.1.2.tar.gz (108 kB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.data import HeteroData

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 2. Data Loading Pipeline
We use the **Telecom Italia Big Data Challenge (Milan)** dataset. The data is processed into sequences of 12 timesteps (2 hours) to predict the next 3 timesteps (30 minutes).

In [ ]:
def load_telecom_italia(data_dir, n_cells=100, seq_len=12, horizon=3):
    files = [f for f in os.listdir(data_dir) if f.endswith('.txt')]
    if not files:
        raise FileNotFoundError(f"No .txt data files found in {data_dir}")
    
    all_data = []
    for file in files:
        filepath = os.path.join(data_dir, file)
        # Dataset columns: squareid, timeinterval, countrycode, smsin, smsout, callin, callout, internet
        df = pd.read_csv(filepath, sep='\t', header=None,
                         names=['grid', 'interval', 'country', 'sms_in', 'sms_out',
                                'call_in', 'call_out', 'internet'])
        
        df = df[df['grid'] <= n_cells].fillna(0)
        df['activity'] = df['internet'] + df['call_in'] + df['call_out']
        
        pivot = df.pivot_table(index='interval', columns='grid',
                               values='activity', fill_value=0)
        all_data.append(pivot)

    full_pivot = pd.concat(all_data).sort_index()
    data = full_pivot.values.astype(np.float32)
    
    scaler = MinMaxScaler()
    data = scaler.fit_transform(data)
    
    X, y = [], []
    for t in range(len(data) - seq_len - horizon):
        X.append(data[t : t+seq_len])
        y.append(data[t+seq_len : t+seq_len+horizon])
    
    X, y = np.stack(X), np.stack(y)
    s1, s2 = int(0.7*len(X)), int(0.8*len(X))
    return (X[:s1], y[:s1]), (X[s1:s2], y[s1:s2]), (X[s2:], y[s2:]), scaler

## 3. Heterogeneous Graph Engineering
Cells are partitioned into **Macro** (top 10% activity), **Femto** (bottom 30%), and **Pico** BS types.

In [ ]:
def build_hetero_graph(data_matrix, k=5, macro_pct=0.10, femto_pct=0.30):
    T, N = data_matrix.shape
    mean_act = data_matrix.mean(axis=0)
    sorted_idx = np.argsort(mean_act)
    
    n_femto = int(N * femto_pct)
    n_macro = int(N * macro_pct)
    
    femto_idx = sorted_idx[:n_femto]
    macro_idx = sorted_idx[-n_macro:]
    pico_idx  = sorted_idx[n_femto:-n_macro]
    
    macro_local = {g:l for l,g in enumerate(macro_idx)}
    pico_local  = {g:l for l,g in enumerate(pico_idx)}
    femto_local = {g:l for l,g in enumerate(femto_idx)}
    
    hetero = HeteroData()
    hetero['macro'].x = torch.tensor(mean_act[macro_idx]).unsqueeze(1).float()
    hetero['pico'].x  = torch.tensor(mean_act[pico_idx]).unsqueeze(1).float()
    hetero['femto'].x = torch.tensor(mean_act[femto_idx]).unsqueeze(1).float()
    
    sqrt_n = int(np.sqrt(N))
    coords = np.array([[i//sqrt_n, i%sqrt_n] for i in range(N)], dtype=np.float32)
    
    def knn_edges(src_ids, dst_ids, k):
        src_test = src_ids[0]
        local_map_src = macro_local if src_test in macro_local else (pico_local if src_test in pico_local else femto_local)
        dst_test = dst_ids[0]
        local_map_dst = macro_local if dst_test in macro_local else (pico_local if dst_test in pico_local else femto_local)
        
        src_nodes, dst_nodes = [], []
        for s_g in src_ids:
            dists = np.linalg.norm(coords[dst_ids] - coords[s_g], axis=1)
            n_to_find = k+1 if np.array_equal(src_ids, dst_ids) else k
            nns = np.argsort(dists)[:n_to_find]
            for d_l in nns:
                d_g = dst_ids[d_l]
                if s_g == d_g: continue
                src_nodes.append(local_map_src[s_g])
                dst_nodes.append(local_map_dst[d_g])
                if len(src_nodes) >= k * len(src_ids): break
        return torch.tensor([src_nodes, dst_nodes], dtype=torch.long)

    hetero['macro','geo','macro'].edge_index = knn_edges(macro_idx, macro_idx, k)
    hetero['pico','geo','pico'].edge_index   = knn_edges(pico_idx, pico_idx, k)
    hetero['femto','geo','femto'].edge_index = knn_edges(femto_idx, femto_idx, k)
    hetero['macro','cross','pico'].edge_index  = knn_edges(macro_idx, pico_idx, k)
    hetero['macro','cross','femto'].edge_index = knn_edges(macro_idx, femto_idx, k)
    return hetero, macro_idx, pico_idx, femto_idx

## 4. TASTF Model Architecture
Combining HeteroGNN for spatial features and Transformer for temporal signals.

In [ ]:
class HeteroGNNEncoder(nn.Module):
    def __init__(self, in_dim=1, hidden=32, out_dim=64):
        super().__init__()
        self.conv1 = HeteroConv({
            ('macro','geo','macro'):   SAGEConv(in_dim, hidden),
            ('pico','geo','pico'):     SAGEConv(in_dim, hidden),
            ('femto','geo','femto'):   SAGEConv(in_dim, hidden),
            ('macro','cross','pico'):  SAGEConv(in_dim, hidden),
            ('macro','cross','femto'): SAGEConv(in_dim, hidden),
        }, aggr='sum')
        self.conv2 = HeteroConv({
            ('macro','geo','macro'):   SAGEConv(hidden, out_dim),
            ('pico','geo','pico'):     SAGEConv(hidden, out_dim),
            ('femto','geo','femto'):   SAGEConv(hidden, out_dim),
            ('macro','cross','pico'):  SAGEConv(hidden, out_dim),
            ('macro','cross','femto'): SAGEConv(hidden, out_dim),
        }, aggr='sum')
        self.norm = nn.LayerNorm(out_dim)
        self.proj = nn.ModuleDict({
            'macro': nn.Linear(out_dim, out_dim),
            'pico':  nn.Linear(out_dim, out_dim),
            'femto': nn.Linear(out_dim, out_dim),
        })
    def forward(self, x_dict, edge_index_dict):
        out = self.conv1(x_dict, edge_index_dict)
        out = {k: F.relu(v) for k, v in out.items()}
        out = self.conv2(out, edge_index_dict)
        out = {k: F.relu(self.norm(v)) for k, v in out.items()}
        return {k: self.proj[k](v) for k, v in out.items()}

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(1))
    def forward(self, x): return x + self.pe[:x.size(0)]

class TASTF(nn.Module):
    def __init__(self, N, gnn_out=64, nhead=4, tf_layers=2, horizon=3,
                 macro_idx=None, pico_idx=None, femto_idx=None):
        super().__init__()
        self.N, self.horizon = N, horizon
        self.macro_idx, self.pico_idx, self.femto_idx = macro_idx, pico_idx, femto_idx
        self.gnn = HeteroGNNEncoder(in_dim=1, hidden=32, out_dim=gnn_out)
        self.pos_enc = PositionalEncoding(gnn_out)
        enc_layer = nn.TransformerEncoderLayer(d_model=gnn_out, nhead=nhead,
                        dim_feedforward=128, batch_first=False)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=tf_layers)
        self.head = nn.Linear(gnn_out, horizon)
    def forward(self, x_seq, hetero_graph):
        B, T, N = x_seq.shape
        spatial_seq = []
        for t in range(T):
            xt = x_seq[:, t, :]
            x_dict = {'macro': xt[:,self.macro_idx].reshape(-1,1),
                      'pico':  xt[:,self.pico_idx].reshape(-1,1),
                      'femto': xt[:,self.femto_idx].reshape(-1,1)}
            emb = self.gnn(x_dict, hetero_graph.edge_index_dict)
            full = torch.zeros(B, N, 64, device=x_seq.device)
            full[:, self.macro_idx] = emb['macro'].view(B, -1, 64)
            full[:, self.pico_idx]  = emb['pico'].view(B, -1, 64)
            full[:, self.femto_idx] = emb['femto'].view(B, -1, 64)
            spatial_seq.append(full)
        seq = torch.stack(spatial_seq, dim=0).view(T, B*N, 64)
        seq = self.pos_enc(seq)
        out = self.transformer(seq)[-1]
        pred = self.head(out).view(B, N, self.horizon).permute(0,2,1)
        return pred

## 5. Training Loop
Running 50 epochs with early stopping (patience=10).

In [ ]:
# Config
DATA_DIR = './wireless dataset'
N_CELLS, SEQ_LEN, HORIZON = 100, 12, 3
BATCH_SIZE, EPOCHS, LR = 32, 50, 1e-3

# Load Data
(X_tr, y_tr), (X_val, y_val), (X_te, y_te), scaler = load_telecom_italia(DATA_DIR, N_CELLS)
X_tr, y_tr = torch.tensor(X_tr).float(), torch.tensor(y_tr).float()
X_val, y_val = torch.tensor(X_val).float(), torch.tensor(y_val).float()
X_te, y_te = torch.tensor(X_te).float(), torch.tensor(y_te).float()

# Build Graph
hetero, macro_idx, pico_idx, femto_idx = build_hetero_graph(X_tr.numpy().reshape(-1, N_CELLS))
hetero = hetero.to(DEVICE)

# Init Model
model = TASTF(N=N_CELLS, horizon=HORIZON, macro_idx=macro_idx, pico_idx=pico_idx, femto_idx=femto_idx).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()
best_val = 1e9
patience, wait = 10, 0

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(X_tr))
    train_loss = 0
    for i in range(0, len(X_tr), BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        xb, yb = X_tr[idx].to(DEVICE), y_tr[idx].to(DEVICE)
        opt.zero_grad()
        pred = model(xb, hetero)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        train_loss += loss.item()
        
    model.eval()
    with torch.no_grad():
        v_pred = model(X_val.to(DEVICE), hetero)
        v_loss = loss_fn(v_pred, y_val.to(DEVICE)).item()
    
    print(f"Epoch {epoch+1:02d} | Train: {train_loss/len(X_tr):.6f} | Val: {v_loss:.6f}")
    if v_loss < best_val:
        best_val = v_loss
        torch.save(model.state_dict(), 'tastf_notebook.pt')
        wait = 0
    else:
        wait += 1
        if wait >= patience: print("Early stopping"); break

## 6. Evaluation & Visualization
Calculate metrics and plot tier-specific predictions.

In [ ]:
model.load_state_dict(torch.load('tastf_notebook.pt'))
model.eval()
with torch.no_grad():
    te_pred = model(X_te.to(DEVICE), hetero).cpu().numpy()
    te_true = y_te.numpy()

mae  = np.mean(np.abs(te_pred - te_true))
rmse = np.sqrt(np.mean((te_pred - te_true)**2))
print(f"Final MAE: {mae:.4f} | RMSE: {rmse:.4f}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_indices = [macro_idx[0], pico_idx[0], femto_idx[0]]
labels = ['Macro BS', 'Pico BS', 'Femto BS']
colors = ['#FF4B4B', '#1C83E1', '#00D67D']

for i, (idx, label, color) in enumerate(zip(plot_indices, labels, colors)):
    ax = axes[i]
    ax.plot(te_true[:100, 0, idx], label='Truth', color='black', alpha=0.3)
    ax.plot(te_pred[:100, 0, idx], '--', label='TASTF', color=color)
    ax.set_title(label)
    ax.legend()
plt.show()